# 02 Clean And Prepare States

Clean Frankfurt Bronze state vectors, standardize key fields, and write the Silver analysis base table.

Primary targets:

- `adsb_airspace_eddf.slv_airspace.flight_states_clean`
- `adsb_airspace_eddf.ref.grid_cells`
- `adsb_airspace_eddf.obs.pipeline_run_log`

## Cleaning Contract

- resolve one Bronze `state_vectors_data4` ingestion run
- optionally narrow to one `hour` partition via `source_hour_epoch`
- standardize timestamps, units, and text fields
- keep the Frankfurt spatial box and the terminal-airspace altitude scope
- derive `time_window_5m`, `grid_id`, and `altitude_band_id`
- append cleaned rows into `slv_airspace.flight_states_clean`
- keep the Silver write idempotent by deleting the same `run_id` before rewrite when requested

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import yaml

from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def ensure_text(value) -> str:
    if value is None:
        return ""
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def parse_float(raw_value: str) -> float:
    return float(raw_value.strip())

def parse_optional_int(raw_value: str) -> int | None:
    text = raw_value.strip()
    if not text:
        return None
    return int(text)

def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {
        field.name: normalize_scalar(payload.get(field.name))
        for field in target_schema
    }
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_source_hour_epoch = ""
default_grid_resolution_deg = "0.05"
default_max_altitude_ft = "24500"
default_include_on_ground = "false"
default_overwrite_source_run = "true"

catalog = default_catalog
source_run_id_raw = default_source_run_id
source_hour_epoch_raw = default_source_hour_epoch
grid_resolution_deg_raw = default_grid_resolution_deg
max_altitude_ft_raw = default_max_altitude_ft
include_on_ground_raw = default_include_on_ground
overwrite_source_run_raw = default_overwrite_source_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("source_hour_epoch", default_source_hour_epoch)
    dbutils.widgets.text("grid_resolution_deg", default_grid_resolution_deg)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("include_on_ground", default_include_on_ground, ["true", "false"])
    dbutils.widgets.dropdown("overwrite_source_run", default_overwrite_source_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    source_hour_epoch_raw = dbutils.widgets.get("source_hour_epoch").strip()
    grid_resolution_deg_raw = dbutils.widgets.get("grid_resolution_deg").strip() or default_grid_resolution_deg
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    include_on_ground_raw = dbutils.widgets.get("include_on_ground")
    overwrite_source_run_raw = dbutils.widgets.get("overwrite_source_run")

focus_airport = region_config["focus_airport"]
bbox = region_config["regional_bbox"]
grid_resolution_deg = parse_float(grid_resolution_deg_raw)
max_altitude_ft = parse_float(max_altitude_ft_raw)
include_on_ground = parse_bool(include_on_ground_raw)
overwrite_source_run = parse_bool(overwrite_source_run_raw)
source_hour_epoch = parse_optional_int(source_hour_epoch_raw)

if grid_resolution_deg <= 0:
    raise ValueError("grid_resolution_deg must be positive")
if max_altitude_ft <= 0:
    raise ValueError("max_altitude_ft must be positive")

bronze_table = f"{catalog}.brz_adsb.hist_state_vectors"
silver_table = f"{catalog}.slv_airspace.flight_states_clean"
grid_ref_table = f"{catalog}.ref.grid_cells"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"
source_run_id_input = source_run_id_raw.strip()


In [ ]:
if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    latest_success_row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.ingestion_log
        WHERE source_object = 'state_vectors_data4'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if latest_success_row is None:
        raise ValueError("No successful state_vectors_data4 Bronze run found. Run 01a_ingest_opensky_history.ipynb first.")
    selected_source_run_id = latest_success_row["run_id"]

bronze_df = spark.table(bronze_table).where(F.col("run_id") == F.lit(selected_source_run_id))
if source_hour_epoch is not None:
    bronze_df = bronze_df.where(F.col("hour") == F.lit(source_hour_epoch))

source_row_count = bronze_df.count()
if source_row_count == 0:
    raise ValueError(
        f"No Bronze rows found for run_id={selected_source_run_id} and source_hour_epoch={source_hour_epoch}."
    )

source_summary_row = bronze_df.agg(
    F.countDistinct("hour").alias("hour_partition_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.min("hour").alias("min_hour"),
    F.max("hour").alias("max_hour")
).first()

run_plan = {
    "catalog": catalog,
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "bronze_table": bronze_table,
    "silver_table": silver_table,
    "grid_ref_table": grid_ref_table,
    "grid_resolution_deg": grid_resolution_deg,
    "max_altitude_ft": max_altitude_ft,
    "include_on_ground": include_on_ground,
    "overwrite_source_run": overwrite_source_run,
    "source_row_count": source_row_count,
    "source_aircraft_count": int(source_summary_row["aircraft_count"]),
    "source_hour_partition_count": int(source_summary_row["hour_partition_count"]),
    "min_hour": int(source_summary_row["min_hour"]),
    "max_hour": int(source_summary_row["max_hour"]),
}

run_plan


In [ ]:
bbox_min_lat = float(bbox["min_latitude"])
bbox_max_lat = float(bbox["max_latitude"])
bbox_min_lon = float(bbox["min_longitude"])
bbox_max_lon = float(bbox["max_longitude"])

meters_to_feet = 3.28083989501312
meters_per_second_to_knots = 1.9438444924406
meters_per_second_to_fpm = 196.850393700787
grid_epsilon = 1e-9

working_df = (
    bronze_df
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("time"))))
    .withColumn("time_window_5m", F.to_timestamp(F.from_unixtime(F.floor(F.col("time") / F.lit(300)) * F.lit(300))))
    .withColumn("callsign_clean", F.when(F.length(F.trim(F.col("callsign"))) > 0, F.trim(F.col("callsign"))))
    .withColumn("latitude", F.col("lat").cast("double"))
    .withColumn("longitude", F.col("lon").cast("double"))
    .withColumn("altitude_ft_raw", F.col("baroaltitude") * F.lit(meters_to_feet))
    .withColumn(
        "altitude_ft",
        F.when(F.col("baroaltitude").isNull(), F.lit(None)).otherwise(F.greatest(F.col("altitude_ft_raw"), F.lit(0.0))),
    )
    .withColumn(
        "speed_kt",
        F.when(F.col("velocity").isNull(), F.lit(None)).otherwise(F.col("velocity") * F.lit(meters_per_second_to_knots)),
    )
    .withColumn(
        "heading_deg",
        F.when((F.col("heading") >= 0) & (F.col("heading") <= 360), F.col("heading")).otherwise(F.lit(None)),
    )
    .withColumn(
        "vertical_rate_fpm",
        F.when(F.col("vertrate").isNull(), F.lit(None)).otherwise(F.col("vertrate") * F.lit(meters_per_second_to_fpm)),
    )
    .where(F.col("icao24").isNotNull())
    .where(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .where(F.col("latitude").between(bbox_min_lat, bbox_max_lat))
    .where(F.col("longitude").between(bbox_min_lon, bbox_max_lon))
    .where(F.col("baroaltitude").isNotNull())
    .where((F.col("velocity").isNull()) | ((F.col("velocity") >= 0) & (F.col("velocity") <= 350)))
)

if not include_on_ground:
    working_df = working_df.where(~F.coalesce(F.col("onground"), F.lit(False)))

working_df = (
    working_df
    .where(F.col("altitude_ft") <= F.lit(max_altitude_ft))
    .withColumn(
        "altitude_band_id",
        F.when(F.col("altitude_ft") <= F.lit(10000.0), F.lit("SFC_FL100")).otherwise(F.lit("FL100_FL245")),
    )
    .withColumn(
        "lat_for_grid",
        F.when(F.col("latitude") >= F.lit(bbox_max_lat), F.lit(bbox_max_lat - grid_epsilon)).otherwise(F.col("latitude")),
    )
    .withColumn(
        "lon_for_grid",
        F.when(F.col("longitude") >= F.lit(bbox_max_lon), F.lit(bbox_max_lon - grid_epsilon)).otherwise(F.col("longitude")),
    )
    .withColumn(
        "lat_index",
        F.floor((F.col("lat_for_grid") - F.lit(bbox_min_lat)) / F.lit(grid_resolution_deg)).cast("int"),
    )
    .withColumn(
        "lon_index",
        F.floor((F.col("lon_for_grid") - F.lit(bbox_min_lon)) / F.lit(grid_resolution_deg)).cast("int"),
    )
    .withColumn("grid_id", F.format_string("fra_%03d_%03d", F.col("lat_index"), F.col("lon_index")))
)

silver_df = working_df.select(
    F.col("event_time"),
    F.col("time_window_5m"),
    F.col("grid_id"),
    F.col("icao24"),
    F.col("callsign_clean").alias("callsign"),
    F.col("latitude"),
    F.col("longitude"),
    F.round(F.col("altitude_ft"), 2).alias("altitude_ft"),
    F.round(F.col("speed_kt"), 2).alias("speed_kt"),
    F.round(F.col("heading_deg"), 4).alias("heading_deg"),
    F.round(F.col("vertical_rate_fpm"), 2).alias("vertical_rate_fpm"),
    F.col("altitude_band_id"),
    F.lit(focus_airport).alias("focus_airport"),
    F.lit(bronze_table).alias("source_table"),
    F.current_timestamp().alias("ingested_at"),
    F.lit(selected_source_run_id).alias("run_id"),
)

grid_cells_df = (
    working_df
    .select("grid_id", "lat_index", "lon_index")
    .distinct()
    .withColumn("min_latitude", F.lit(bbox_min_lat) + F.col("lat_index") * F.lit(grid_resolution_deg))
    .withColumn("max_latitude", F.col("min_latitude") + F.lit(grid_resolution_deg))
    .withColumn("min_longitude", F.lit(bbox_min_lon) + F.col("lon_index") * F.lit(grid_resolution_deg))
    .withColumn("max_longitude", F.col("min_longitude") + F.lit(grid_resolution_deg))
    .withColumn("center_latitude", (F.col("min_latitude") + F.col("max_latitude")) / F.lit(2.0))
    .withColumn("center_longitude", (F.col("min_longitude") + F.col("max_longitude")) / F.lit(2.0))
    .withColumn("grid_resolution_deg", F.lit(grid_resolution_deg))
    .withColumn("active_flag", F.lit(True))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        "grid_id",
        "min_latitude",
        "max_latitude",
        "min_longitude",
        "max_longitude",
        "center_latitude",
        "center_longitude",
        "grid_resolution_deg",
        "active_flag",
        "created_at",
        "run_id",
    )
)

quality_row = silver_df.agg(
    F.count("*").alias("clean_row_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.countDistinct("grid_id").alias("grid_count"),
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time"),
    F.min("altitude_ft").alias("min_altitude_ft"),
    F.max("altitude_ft").alias("max_altitude_ft"),
).first()

clean_summary = {
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "source_row_count": source_row_count,
    "clean_row_count": int(quality_row["clean_row_count"]),
    "aircraft_count": int(quality_row["aircraft_count"]),
    "grid_count": int(quality_row["grid_count"]),
    "min_event_time": str(quality_row["min_event_time"]),
    "max_event_time": str(quality_row["max_event_time"]),
    "min_altitude_ft": float(quality_row["min_altitude_ft"]),
    "max_altitude_ft": float(quality_row["max_altitude_ft"]),
}

if clean_summary["clean_row_count"] == 0:
    raise ValueError("Cleaning removed all rows. Relax the filters or inspect the Bronze source window.")

clean_summary


In [ ]:
pipeline_started_at = utc_now_naive()
pipeline_status = "success"
pipeline_error_message = None

try:
    grid_cells_df.createOrReplaceTempView("grid_cells_upsert")
    spark.sql(
        f"""
        MERGE INTO {grid_ref_table} AS target
        USING grid_cells_upsert AS source
        ON target.grid_id = source.grid_id
        WHEN MATCHED THEN UPDATE SET
          target.min_latitude = source.min_latitude,
          target.max_latitude = source.max_latitude,
          target.min_longitude = source.min_longitude,
          target.max_longitude = source.max_longitude,
          target.center_latitude = source.center_latitude,
          target.center_longitude = source.center_longitude,
          target.grid_resolution_deg = source.grid_resolution_deg,
          target.active_flag = source.active_flag,
          target.created_at = source.created_at,
          target.run_id = source.run_id
        WHEN NOT MATCHED THEN INSERT (
          grid_id,
          min_latitude,
          max_latitude,
          min_longitude,
          max_longitude,
          center_latitude,
          center_longitude,
          grid_resolution_deg,
          active_flag,
          created_at,
          run_id
        ) VALUES (
          source.grid_id,
          source.min_latitude,
          source.max_latitude,
          source.min_longitude,
          source.max_longitude,
          source.center_latitude,
          source.center_longitude,
          source.grid_resolution_deg,
          source.active_flag,
          source.created_at,
          source.run_id
        )
        """
    )

    if overwrite_source_run:
        spark.sql(f"DELETE FROM {silver_table} WHERE run_id = '{selected_source_run_id}'")

    silver_df.write.mode("append").saveAsTable(silver_table)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=pipeline_log_table,
        payload={
            "run_id": selected_source_run_id,
            "pipeline_name": "02_clean_and_prepare_states",
            "layer": "silver",
            "target_table": silver_table,
            "status": pipeline_status,
            "rows_read": source_row_count,
            "rows_written": clean_summary["clean_row_count"] if pipeline_status == "success" else 0,
            "started_at": pipeline_started_at,
            "completed_at": utc_now_naive(),
            "error_message": pipeline_error_message,
            "metadata_json": json.dumps(
                {
                    "source_run_id": selected_source_run_id,
                    "source_hour_epoch": source_hour_epoch,
                    "grid_resolution_deg": grid_resolution_deg,
                    "max_altitude_ft": max_altitude_ft,
                    "include_on_ground": include_on_ground,
                },
                sort_keys=True,
                default=str,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "silver_table": silver_table,
    "grid_ref_table": grid_ref_table,
    "source_row_count": source_row_count,
    "clean_row_count": clean_summary["clean_row_count"],
    "aircraft_count": clean_summary["aircraft_count"],
    "grid_count": clean_summary["grid_count"],
}

final_summary
